# Notebook title

description here

### Import Libraries

fill this out later

In [1]:
import time
import requests
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import scheduleleaguev2

pd.set_option('display.max_columns', None)

SEASON = "2025-26"
REQUEST_DELAY_SECONDS = 0.25
BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
DATA = "../data/2025_raw_box_score_stats.csv"


### Consolidate games 

fill this out later

In [2]:
def game_consolidation(season_data):
    '''
    Fill this out later
    '''
    df = pd.read_csv(season_data)
    df = df.drop(columns=[
                "TEAM_ID", "SEASON_ID", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
                "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
            ])

    home = df[df["MATCHUP"].str.contains("vs.")]
    away = df[df["MATCHUP"].str.contains("@")]
    games = home.merge(away, on="GAME_ID", suffixes=("_HOME", "_AWAY"))
    games = games.rename(columns={"MATCHUP_AWAY": 'MATCHUP'})
    games['WINNER'] = np.where(games['WL_HOME'] == "W", games['TEAM_NAME_HOME'], games['TEAM_NAME_AWAY'])

    games = games[[
            "GAME_ID", "GAME_DATE_HOME", "MATCHUP", 
            "TEAM_ABBREVIATION_AWAY", "TEAM_NAME_AWAY", "PTS_AWAY",
            "TEAM_ABBREVIATION_HOME", "TEAM_NAME_HOME", "PTS_HOME", "WINNER"
        ]]

    return games


test = game_consolidation(DATA)
test


,GAME_ID,GAME_DATE_HOME,MATCHUP,TEAM_ABBREVIATION_AWAY,TEAM_NAME_AWAY,PTS_AWAY,TEAM_ABBREVIATION_HOME,TEAM_NAME_HOME,PTS_HOME,WINNER
0,22500001,2025-10-21,HOU @ OKC,HOU,Houston Rockets,124,OKC,Oklahoma City Thunder,125,Oklahoma City Thunder
1,22500002,2025-10-21,GSW @ LAL,GSW,Golden State Warriors,119,LAL,Los Angeles Lakers,109,Golden State Warriors
2,22500086,2025-10-22,WAS @ MIL,WAS,Washington Wizards,120,MIL,Milwaukee Bucks,133,Milwaukee Bucks
3,22500003,2025-10-22,CLE @ NYK,CLE,Cleveland Cavaliers,111,NYK,New York Knicks,119,New York Knicks
4,22500081,2025-10-22,MIA @ ORL,MIA,Miami Heat,121,ORL,Orlando Magic,125,Orlando Magic
...,...,...,...,...,...,...,...,...,...,...
1220,22501194,2026-04-12,MEM @ HOU,MEM,Memphis Grizzlies,101,HOU,Houston Rockets,132,Houston Rockets
1221,22501199,2026-04-12,GSW @ LAC,GSW,Golden State Warriors,110,LAC,LA Clippers,115,LA Clippers
1222,22501198,2026-04-12,UTA @ LAL,UTA,Utah Jazz,107,LAL,Los Angeles Lakers,131,Los Angeles Lakers
1223,22501189,2026-04-12,ATL @ MIA,ATL,Atlanta Hawks,117,MIA,Miami Heat,143,Miami Heat


### Get the game start times

fill this out later


In [3]:
def nba_game_start_times(year):
    '''
    Fill this out later
    '''
    sched = scheduleleaguev2.ScheduleLeagueV2(season=year)
    df = sched.season_games.get_data_frame()

    df = df[["gameId", "gameDate", "gameDateTimeUTC","gameDateTimeEst",
            "gameStatusText", "homeTeam_teamTricode", "awayTeam_teamTricode", "gameLabel"]]

    df = df.rename(columns={"gameId": 'GAME_ID'})
    df["GAME_ID"] = pd.to_numeric(df["GAME_ID"], errors="coerce").astype("Int64")
    games_to_keep = ['', 'Emirates NBA Cup', 'NBA Mexico City Game', 'NBA Berlin Game', 'NBA London Game', 'AWS NBA Rivals Week','NBA Pioneers Classic']
    df_filtered = df.loc[df['gameLabel'].isin(games_to_keep)]

    return df_filtered

test2 = nba_game_start_times(SEASON)
test2


,GAME_ID,gameDate,gameDateTimeUTC,gameDateTimeEst,gameStatusText,homeTeam_teamTricode,awayTeam_teamTricode,gameLabel
71,22500001,10/21/2025 00:00:00,2025-10-21T23:30:00Z,2025-10-21T19:30:00Z,Final/OT2,OKC,HOU,
72,22500002,10/21/2025 00:00:00,2025-10-22T02:00:00Z,2025-10-21T22:00:00Z,Final,LAL,GSW,
73,22500003,10/22/2025 00:00:00,2025-10-22T23:00:00Z,2025-10-22T19:00:00Z,Final,NYK,CLE,
74,22500004,10/22/2025 00:00:00,2025-10-23T01:30:00Z,2025-10-22T21:30:00Z,Final,DAL,SAS,
75,22500080,10/22/2025 00:00:00,2025-10-22T23:00:00Z,2025-10-22T19:00:00Z,Final,CHA,BKN,
...,...,...,...,...,...,...,...,...
1304,22501196,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,OKC,PHX,
1305,22501197,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,SAS,DEN,
1306,22501198,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,LAL,UTA,
1307,22501199,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,LAC,GSW,


### Merge DF's and create columns with Kalshi info

fill this out later

In [4]:

def games_with_kalshi_info_converter(df1, df2):
    '''
    Fill this out later
    '''
    # merge the two dfs on GAME_ID using inner
    merged = pd.merge(df1, df2, on="GAME_ID", how="inner")
    
    # rename columns for clarity
    merged = merged.rename(columns={
        "GAME_DATE_HOME": "GAME_DATE",
        "TEAM_ABBREVIATION_AWAY": "ABBR_AWAY",
        "TEAM_NAME_AWAY": "NAME_AWAY",
        "TEAM_ABBREVIATION_HOME": "ABBR_HOME",
        "TEAM_NAME_HOME": "NAME_HOME",
        "gameStatusText": "STATUS"
    })

    # convert gameDateTimeEst column from object to datetieme
    merged["gameDateTimeEst"] = pd.to_datetime(merged["gameDateTimeEst"])

    # convert object columns to strings
    merged["ABBR_AWAY"] = merged["ABBR_AWAY"].astype(str)
    merged["ABBR_HOME"] = merged["ABBR_HOME"].astype(str)

    # create new column for clarity using 12 hour clock
    merged["GAME_START(EST)"] = (merged["gameDateTimeEst"].dt.strftime("%I:%M%p").str.lstrip("0"))

    # create Kalshi market ID's for away teams
    merged['KALSHI_ID_AWAY'] = ('KXNBAGAME-' +
        merged['gameDateTimeEst'].dt.strftime('%y') +
        merged['gameDateTimeEst'].dt.strftime('%b').str.upper() +
        merged['gameDateTimeEst'].dt.strftime('%d') + 
        merged['ABBR_AWAY'] +
        merged['ABBR_HOME'] + '-' +
        merged['ABBR_AWAY']
    )

    # create Kalshi market ID's for home teams
    merged['KALSHI_ID_HOME'] = ('KXNBAGAME-' + 
        merged['gameDateTimeEst'].dt.strftime('%y') +
        merged['gameDateTimeEst'].dt.strftime('%b').str.upper() +
        merged['gameDateTimeEst'].dt.strftime('%d') + 
        merged['ABBR_AWAY'] +
        merged['ABBR_HOME'] + '-' +
        merged['ABBR_HOME']
    )

    # create a timestamp column that will be used for fetching Kalshi data later
    merged['TIMESTAMP'] = (merged['gameDateTimeEst'].astype('int64') // 1_000_000_000)

    # reorganize final column order
    merged = merged[[
        "GAME_ID", "GAME_DATE", "GAME_START(EST)", "TIMESTAMP", 
        "KALSHI_ID_AWAY", "KALSHI_ID_HOME", "MATCHUP", 
        "ABBR_AWAY", "NAME_AWAY", "PTS_AWAY", 
        "ABBR_HOME", "NAME_HOME", "PTS_HOME",
        "WINNER", "STATUS"
    ]]

    return merged


test3 = games_with_kalshi_info_converter(test, test2)
test3

,GAME_ID,GAME_DATE,GAME_START(EST),TIMESTAMP,KALSHI_ID_AWAY,KALSHI_ID_HOME,MATCHUP,ABBR_AWAY,NAME_AWAY,PTS_AWAY,ABBR_HOME,NAME_HOME,PTS_HOME,WINNER,STATUS
0,22500001,2025-10-21,7:30PM,1761075000,KXNBAGAME-25OCT21HOUOKC-HOU,KXNBAGAME-25OCT21HOUOKC-OKC,HOU @ OKC,HOU,Houston Rockets,124,OKC,Oklahoma City Thunder,125,Oklahoma City Thunder,Final/OT2
1,22500002,2025-10-21,10:00PM,1761084000,KXNBAGAME-25OCT21GSWLAL-GSW,KXNBAGAME-25OCT21GSWLAL-LAL,GSW @ LAL,GSW,Golden State Warriors,119,LAL,Los Angeles Lakers,109,Golden State Warriors,Final
2,22500086,2025-10-22,8:00PM,1761163200,KXNBAGAME-25OCT22WASMIL-WAS,KXNBAGAME-25OCT22WASMIL-MIL,WAS @ MIL,WAS,Washington Wizards,120,MIL,Milwaukee Bucks,133,Milwaukee Bucks,Final
3,22500003,2025-10-22,7:00PM,1761159600,KXNBAGAME-25OCT22CLENYK-CLE,KXNBAGAME-25OCT22CLENYK-NYK,CLE @ NYK,CLE,Cleveland Cavaliers,111,NYK,New York Knicks,119,New York Knicks,Final
4,22500081,2025-10-22,7:00PM,1761159600,KXNBAGAME-25OCT22MIAORL-MIA,KXNBAGAME-25OCT22MIAORL-ORL,MIA @ ORL,MIA,Miami Heat,121,ORL,Orlando Magic,125,Orlando Magic,Final
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1220,22501194,2026-04-12,8:30PM,1776025800,KXNBAGAME-26APR12MEMHOU-MEM,KXNBAGAME-26APR12MEMHOU-HOU,MEM @ HOU,MEM,Memphis Grizzlies,101,HOU,Houston Rockets,132,Houston Rockets,Final
1221,22501199,2026-04-12,8:30PM,1776025800,KXNBAGAME-26APR12GSWLAC-GSW,KXNBAGAME-26APR12GSWLAC-LAC,GSW @ LAC,GSW,Golden State Warriors,110,LAC,LA Clippers,115,LA Clippers,Final
1222,22501198,2026-04-12,8:30PM,1776025800,KXNBAGAME-26APR12UTALAL-UTA,KXNBAGAME-26APR12UTALAL-LAL,UTA @ LAL,UTA,Utah Jazz,107,LAL,Los Angeles Lakers,131,Los Angeles Lakers,Final
1223,22501189,2026-04-12,6:00PM,1776016800,KXNBAGAME-26APR12ATLMIA-ATL,KXNBAGAME-26APR12ATLMIA-MIA,ATL @ MIA,ATL,Atlanta Hawks,117,MIA,Miami Heat,143,Miami Heat,Final


In [ ]:
test3.to_csv("../data/_main(all-games).csv", index=False)


In [9]:
game_dict = dict(zip(test3['KALSHI_ID_AWAY'], test3['TIMESTAMP'])) | dict(zip(test3['KALSHI_ID_HOME'], test3['TIMESTAMP']))
#game_dict

### Kalshi Pre Game Data Fetch

fill this out later


In [6]:
def fetch_kalshi_candlesticks(ticker, end_ts, period_interval=60):
    """
    Use this funciton to fetch historical candlestick data from the Kalshi API.

    PARAMETERS;
        - ticker (str): market ticker symbol ('KXNBAGAME-25OCT21GSWLAL-GSW')
        - end_ts (int): timestamp marking the ending of the desired period (start of the game)
        - period_interval (int): time period length of each candlestick in minutes (60=1hr candles, 1=1min candles)

    Returns; a DataFrame containing the candlestick data with the following columns (among others):
        - KALSHI_ID (str): the unique market identifier
        - TIMESTAMP (datetime): date and time of candlestick ending
        - end_period_ts (int): timestamp of the candle close
        - open_interest (str): number of contracts still open by end of the candlestick period
        - volume (str): number of contracts bought on the market during the candlestick period
        - price_close (float): price of the last trade during the candlestick period
        - price_high (float): highest trade price during the candlestick period
        - price_low (float): lowest trade price during the candlestick period
        - price_mean (float): volume weighted average price during the candlestick period
        - price_open (float): price of the first trade during the candlestick period
    """
    # collect one hour of data before game starts (and exclude previous hours candle)
    period_seconds = period_interval * 60
    start_ts = end_ts - period_seconds + 1

    # list that will collect candlestick dicts from every chunk
    all_candles = []

    # send http get requests to the historical candlesticks endpoint
    response = requests.get(
        f"{BASE_URL}/historical/markets/{ticker}/candlesticks",
        params={'start_ts': start_ts, 'end_ts': end_ts, 'period_interval': period_interval},
        timeout=10)

    # error handling for if the market doesn't exist (or ticker info is wrong), or any other error
    if response.status_code == 404:
        print(f"Market {ticker} not found (404) - skipping")
        return pd.DataFrame()
    response.raise_for_status()

    # parse the JSON response and append this chunk’s candles to our list
    all_candles.extend(response.json()['candlesticks'])

    # necessary pause between requests to avoid Kalshi API rate limits
    time.sleep(REQUEST_DELAY_SECONDS)

    # flatten the nested candle dicts into a table; nested keys joined sep
    df = pd.json_normalize(all_candles, sep='_')

    # add ticker and datetime columns 
    df.insert(0, 'KALSHI_ID', ticker)
    df.insert(1, 'TIMESTAMP', pd.to_datetime(df['end_period_ts'], unit='s', utc=True))

    # keep only the last-hour candle (one row per ticker)
    df = df.sort_values("end_period_ts").tail(1).reset_index(drop=True)

    # drop unnecessary columns
    df = df.drop(columns=[
        "price_previous", "yes_ask_close", "yes_ask_high", "yes_ask_low", 
        "yes_ask_open", "yes_bid_close", "yes_bid_high", "yes_bid_low", "yes_bid_open"
    ])

    # convert columns to float
    float_cols = [
        "open_interest", "volume", "price_close", "price_high",
        "price_low", "price_mean", "price_open",
    ]
    df[float_cols] = df[float_cols].apply(pd.to_numeric, errors="coerce").round(4)

    return df


def kalshi_price_df_generator(game_dict):
    '''
    fill this out later
    '''
    count = 0
    all_dfs = []
    for ticker, ts in game_dict.items():
        df = fetch_kalshi_candlesticks(ticker, ts)
        if df.empty:
            print(f"-> skipped {ticker}")
            continue
        count += 1
        print(f"-> row {count} || {ticker} done")
        all_dfs.append(df)

    combined = pd.concat(all_dfs, ignore_index=True)

    return combined


In [7]:
test4 = kalshi_price_df_generator(game_dict)
test4


-> row 1 || KXNBAGAME-25OCT21HOUOKC-HOU done
-> row 2 || KXNBAGAME-25OCT21GSWLAL-GSW done
-> row 3 || KXNBAGAME-25OCT22WASMIL-WAS done
-> row 4 || KXNBAGAME-25OCT22CLENYK-CLE done
-> row 5 || KXNBAGAME-25OCT22MIAORL-MIA done
-> row 6 || KXNBAGAME-25OCT22SACPHX-SAC done
-> row 7 || KXNBAGAME-25OCT22MINPOR-MIN done
-> row 8 || KXNBAGAME-25OCT22TORATL-TOR done
-> row 9 || KXNBAGAME-25OCT22PHIBOS-PHI done
-> row 10 || KXNBAGAME-25OCT22DETCHI-DET done
-> row 11 || KXNBAGAME-25OCT22SASDAL-SAS done
-> row 12 || KXNBAGAME-25OCT22LACUTA-LAC done
-> row 13 || KXNBAGAME-25OCT22NOPMEM-NOP done
-> row 14 || KXNBAGAME-25OCT22BKNCHA-BKN done
-> row 15 || KXNBAGAME-25OCT23OKCIND-OKC done
-> row 16 || KXNBAGAME-25OCT23DENGSW-DEN done
-> row 17 || KXNBAGAME-25OCT24CLEBKN-CLE done
-> row 18 || KXNBAGAME-25OCT24BOSNYK-BOS done
-> row 19 || KXNBAGAME-25OCT24ATLORL-ATL done
-> row 20 || KXNBAGAME-25OCT24GSWPOR-GSW done
-> row 21 || KXNBAGAME-25OCT24UTASAC-UTA done
-> row 22 || KXNBAGAME-25OCT24SASNOP-SAS do

,KALSHI_ID,TIMESTAMP,end_period_ts,open_interest,volume,price_close,price_high,price_low,price_mean,price_open
0,KXNBAGAME-25OCT21HOUOKC-HOU,2025-10-21 19:00:00+00:00,1761073200,625472.0,96317.0,0.31,0.31,0.30,0.3098,0.31
1,KXNBAGAME-25OCT21GSWLAL-GSW,2025-10-21 22:00:00+00:00,1761084000,605829.0,106223.0,0.59,0.59,0.58,0.5891,0.59
2,KXNBAGAME-25OCT22WASMIL-WAS,2025-10-22 20:00:00+00:00,1761163200,112969.0,6784.0,0.20,0.20,0.19,0.1998,0.20
3,KXNBAGAME-25OCT22CLENYK-CLE,2025-10-22 19:00:00+00:00,1761159600,626019.0,181277.0,0.54,0.60,0.53,0.5537,0.59
4,KXNBAGAME-25OCT22MIAORL-MIA,2025-10-22 19:00:00+00:00,1761159600,276332.0,102947.0,0.24,0.25,0.23,0.2454,0.25
...,...,...,...,...,...,...,...,...,...,...
2443,KXNBAGAME-26APR12MEMHOU-HOU,2026-04-12 20:00:00+00:00,1776024000,46522.0,11249.0,0.90,0.90,0.88,0.8949,0.89
2444,KXNBAGAME-26APR12GSWLAC-LAC,2026-04-12 20:00:00+00:00,1776024000,68095.0,5267.0,0.70,0.70,0.70,0.7000,0.70
2445,KXNBAGAME-26APR12UTALAL-LAL,2026-04-12 20:00:00+00:00,1776024000,150827.0,8361.0,0.88,0.89,0.88,0.8898,0.89
2446,KXNBAGAME-26APR12ATLMIA-MIA,2026-04-12 18:00:00+00:00,1776016800,342637.0,130803.0,0.76,0.77,0.65,0.7300,0.65


In [ ]:
test4.to_csv("../data/kalshi(pre-game).csv", index=False)


### Kalshi In-Game Fetch

fill this out later
